# 📓 Notebook 18 — LangChain Core---**Phase 7 · Industry Frameworks**> LangChain is the most widely adopted framework for building LLM applications. If you're interviewing for an AI/ML role, you **will** be asked about it.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| LCEL (LangChain Expression Language) | Composable chain syntax used in production || Chat models & prompts | LangChain's abstraction over LLM providers || Output parsers | Structured output via LangChain || Chains | Sequential LLM pipelines || Tools & agents | LangChain's agent framework || Memory | Conversation memory in LangChain |### 🏗️ Build: Multi-Model Chatbot with LangChain> **Prerequisite:** `pip install langchain langchain-openai langchain-google-genai langchain-anthropic`

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode, get_langchain_llm

# LangChain LLM — auto-routes through proxy when available
llm = get_langchain_llm()

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")
print(f"   LangChain LLM: {llm.model_name}")

---## 2. Chat Models — The Universal Interface### LangChain vs Raw API| Feature | Raw API (openai, anthropic) | LangChain ||---------|---------------------------|-----------|| Interface | Different per provider | Unified `ChatModel` || Switching models | Rewrite code | Change 1 line || Streaming | Provider-specific | `.stream()` on all || Batch | Manual loop | `.batch()` built-in || Callbacks | Roll your own | Built-in system |> **Interview Tip:** LangChain's value is **abstraction** — write once, run on any provider. This is critical for production systems where you might switch from OpenAI to Anthropic.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage# Basic invocation — same API regardless of providermessages = [    SystemMessage(content="You are a helpful AI assistant specializing in Python."),    HumanMessage(content="What are Python decorators? Explain in 2 sentences."),]response = llm.invoke(messages)print(f"📝 Response: {response.content}")print(f"📊 Token usage: {response.response_metadata.get('token_usage', 'N/A')}")

In [ ]:
# Batch processing — send multiple requests efficientlybatch_messages = [    [HumanMessage(content="What is a list comprehension?")],    [HumanMessage(content="What is a generator?")],    [HumanMessage(content="What is a context manager?")],]results = llm.batch(batch_messages)for i, r in enumerate(results):    print(f"  Q{i+1}: {r.content[:80]}...")

In [ ]:
# Streaming — token by tokenprint("📡 Streaming: ", end="")for chunk in llm.stream([HumanMessage(content="Define 'polymorphism' in 1 sentence.")]):    print(chunk.content, end="", flush=True)print()

---## 3. Prompt Templates — Reusable, Parameterized Prompts

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder# Simple templatesimple_prompt = ChatPromptTemplate.from_messages([    ("system", "You are an expert in {topic}. Be concise."),    ("human", "{question}"),])# Format and invokechain_result = llm.invoke(simple_prompt.format_messages(    topic="machine learning",    question="What is gradient descent?"))print(f"📝 {chain_result.content[:150]}")

In [ ]:
# Template with conversation historychat_prompt = ChatPromptTemplate.from_messages([    ("system", "You are a {role}. Always respond in {style} style."),    MessagesPlaceholder(variable_name="history"),    ("human", "{input}"),])formatted = chat_prompt.format_messages(    role="Python tutor",    style="concise and practical",    history=[        HumanMessage(content="What is a class?"),        AIMessage(content="A class is a blueprint for creating objects with shared attributes and methods."),    ],    input="How do I inherit from one?")result = llm.invoke(formatted)print(f"📝 {result.content[:200]}")

---## 4. LCEL — LangChain Expression Language### The Pipe Operator `|`LCEL lets you compose chains using the pipe operator, like Unix pipes.```pythonchain = prompt | llm | parser```This is the **modern** way to build LangChain applications.

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParserfrom langchain_core.prompts import ChatPromptTemplate# Simple LCEL chainprompt = ChatPromptTemplate.from_messages([    ("system", "You are a helpful assistant."),    ("human", "{input}"),])chain = prompt | llm | StrOutputParser()# Invoke the chainresult = chain.invoke({"input": "What is LCEL in LangChain? One sentence."})print(f"📝 {result}")

In [ ]:
# LCEL with structured JSON outputfrom langchain_core.pydantic_v1 import BaseModel, Fieldfrom typing import Listjson_prompt = ChatPromptTemplate.from_messages([    ("system", "Extract information and return as JSON with keys: topic, summary, difficulty (easy/medium/hard), keywords (list of strings)."),    ("human", "{input}"),])json_chain = json_prompt | llm | JsonOutputParser()result = json_chain.invoke({"input": "Explain how neural networks learn through backpropagation"})print(f"📝 Parsed JSON: {json.dumps(result, indent=2)}")

In [ ]:
# Chaining multiple steps with RunnablePassthroughfrom langchain_core.runnables import RunnablePassthrough, RunnableLambda# Chain: Generate outline → Expand first sectionoutline_prompt = ChatPromptTemplate.from_messages([    ("human", "Create a 3-point outline for an article about {topic}. Return just the outline."),])expand_prompt = ChatPromptTemplate.from_messages([    ("human", "Given this outline:\n{outline}\n\nExpand the first point into a short paragraph."),])# Multi-step chainoutline_chain = outline_prompt | llm | StrOutputParser()full_chain = (    {"topic": RunnablePassthrough()}    | RunnableLambda(lambda x: {"outline": outline_chain.invoke(x)})    | expand_prompt    | llm    | StrOutputParser())result = full_chain.invoke("the future of AI agents")print(f"📝 {result[:300]}...")

---## 5. Output Parsers

In [ ]:
from langchain_core.output_parsers import PydanticOutputParserfrom pydantic import BaseModel, Fieldfrom typing import List, Optional# Define output schemaclass MovieReview(BaseModel):    title: str = Field(description="Movie title")    rating: float = Field(description="Rating out of 10")    pros: List[str] = Field(description="Positive aspects")    cons: List[str] = Field(description="Negative aspects")    recommendation: str = Field(description="Who would enjoy this movie")parser = PydanticOutputParser(pydantic_object=MovieReview)review_prompt = ChatPromptTemplate.from_messages([    ("system", "You are a movie critic. {format_instructions}"),    ("human", "Review the movie: {movie}"),])review_chain = review_prompt | llm | parserresult = review_chain.invoke({    "movie": "Inception",    "format_instructions": parser.get_format_instructions()})print(f"🎬 {result.title}")print(f"⭐ {result.rating}/10")print(f"👍 Pros: {result.pros}")print(f"👎 Cons: {result.cons}")print(f"🎯 For: {result.recommendation}")

---## 6. LangChain Tools & Agents

In [ ]:
from langchain_core.tools import toolfrom langchain.agents import create_tool_calling_agent, AgentExecutorimport math@tooldef calculator(expression: str) -> str:    """Calculate a mathematical expression. Use for any math computation."""    try:        safe = {"sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,                 "pi": math.pi, "e": math.e, "abs": abs, "pow": pow, "log": math.log}        result = eval(expression.replace("^", "**"), {"__builtins__": {}}, safe)        return f"Result: {result}"    except Exception as e:        return f"Error: {e}"@tooldef word_count(text: str) -> str:    """Count the number of words in the given text."""    return f"Word count: {len(text.split())}"@tooldef reverse_string(text: str) -> str:    """Reverse the given text string."""    return f"Reversed: {text[::-1]}"tools = [calculator, word_count, reverse_string]# Create agent with tool callingagent_prompt = ChatPromptTemplate.from_messages([    ("system", "You are a helpful assistant. Use the provided tools when needed."),    ("human", "{input}"),    MessagesPlaceholder(variable_name="agent_scratchpad"),])agent = create_tool_calling_agent(llm, tools, agent_prompt)executor = AgentExecutor(agent=agent, tools=tools, verbose=True)# Testprint("🤖 LangChain Agent Demo")print("=" * 50)result = executor.invoke({"input": "What is the square root of 1764, and how many words are in 'LangChain is an amazing framework for LLM applications'?"})print(f"\n📝 {result['output']}")

---## 7. LangChain Memory

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistoryfrom langchain_core.runnables.history import RunnableWithMessageHistory# Setup store for session historiesstore = {}def get_session_history(session_id: str):    if session_id not in store:        store[session_id] = InMemoryChatMessageHistory()    return store[session_id]# Chain with memorymemory_prompt = ChatPromptTemplate.from_messages([    ("system", "You are a friendly assistant. Remember what the user tells you."),    MessagesPlaceholder(variable_name="history"),    ("human", "{input}"),])memory_chain = memory_prompt | llm | StrOutputParser()chain_with_history = RunnableWithMessageHistory(    memory_chain,    get_session_history,    input_messages_key="input",    history_messages_key="history",)# Test conversation memoryconfig = {"configurable": {"session_id": "user_123"}}msgs = [    "Hi, my name is Abhishek and I work in fintech.",    "I'm learning about AI agents.",    "What's my name and what am I learning about?",]print("🧠 LangChain Memory Demo")print("=" * 50)for msg in msgs:    print(f"\n👤 {msg}")    r = chain_with_history.invoke({"input": msg}, config=config)    print(f"🤖 {r}")

---## 📝 Interview Quiz — LangChain Core### Q1: What is LCEL? Why is it preferred over the legacy chain syntax?<details><summary>💡 Answer</summary>**LCEL (LangChain Expression Language)** uses the pipe operator (`|`) to compose chains:```pythonchain = prompt | llm | parser```**Why it's preferred:**1. **Streaming support** — First-class, automatic streaming through every step2. **Async support** — `.ainvoke()`, `.abatch()` work out of the box3. **Composability** — Chains are Runnables; nest and combine freely4. **Parallelism** — Built-in parallel execution with `RunnableParallel`5. **Retries/fallbacks** — `.with_retry()`, `.with_fallback()` built-in6. **Tracing** — Automatic LangSmith integration at every step**Legacy:** `LLMChain`, `SequentialChain` are deprecated in favor of LCEL.</details>### Q2: How does LangChain abstract across LLM providers?<details><summary>💡 Answer</summary>All providers implement the same `BaseChatModel` interface:- `.invoke(messages)` — Single call- `.batch(messages_list)` — Batch processing- `.stream(messages)` — Token-by-token streaming- `.ainvoke()` — Async invocation**Switching providers:**```python# Only change this one line:llm = ChatOpenAI(model="gpt-4o")llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")llm = ChatAnthropic(model="claude-3-5-sonnet-20241022")```All downstream chains work unchanged. This is LangChain's core value proposition.</details>### Q3: What are the different ways to get structured output in LangChain?<details><summary>💡 Answer</summary>1. **`JsonOutputParser`** — Parse LLM output as JSON2. **`PydanticOutputParser`** — Parse into Pydantic models with validation3. **`.with_structured_output(schema)`** — Model-native structured output (uses function calling under the hood)4. **`OutputFixingParser`** — Auto-fix malformed output by calling LLM again5. **Prompt-based** — Include format instructions in prompt**Recommended:** `.with_structured_output()` — Most reliable, uses tool calling.</details>### Q4: What is a Runnable in LangChain?<details><summary>💡 Answer</summary>A `Runnable` is the fundamental building block in LCEL. Any object that implements:- `.invoke(input)` — Synchronous execution- `.stream(input)` — Streaming execution- `.batch(inputs)` — Batch executionAll LangChain components are Runnables: prompts, LLMs, parsers, retrievers, tools.**Key Runnables:**- `RunnablePassthrough` — Pass input through unchanged- `RunnableLambda` — Wrap any function as a Runnable- `RunnableParallel` — Run multiple Runnables in parallel- `RunnableBranch` — Conditional routing</details>### Q5: LangChain vs LiteLLM — when to use each?<details><summary>💡 Answer</summary>| Aspect | LangChain | LiteLLM ||--------|-----------|---------|| Scope | Full framework (chains, agents, RAG) | LLM API wrapper only || Complexity | Higher learning curve | Minimal, drop-in || Abstractions | Prompts, parsers, agents, memory | Just `completion()` || When to use | Building complex agent systems | Simple multi-model API calls || Overhead | More dependencies, larger footprint | Lightweight |**Rule:** Use LiteLLM for simple LLM calls. Use LangChain when you need the full framework (chains, agents, RAG, memory, tracing).</details>

---## ✅ Notebook 18 Summary| Concept | Key Takeaway ||---------|-------------|| Chat Models | Unified interface across OpenAI, Gemini, Anthropic || LCEL | `prompt \| llm \| parser` — modern chain composition || Prompt Templates | Reusable, parameterized, support chat history || Output Parsers | JSON, Pydantic, auto-fixing for structured output || Agents | `@tool` decorator + `AgentExecutor` for tool-using agents || Memory | `RunnableWithMessageHistory` for conversation persistence |### ➡️ Next: [Notebook 19 — LangChain RAG](./19_langchain_rag.ipynb)